# Threshold Tuning — LDA / NMF / BERTopic
Grid-search over **Absolute** and **Relative** thresholds to maximise multi-label **F1-Score**.
Outputs an annotated heatmap PNG highlighting the optimal parameter pair.

## 1. Installation
Run once to install required dependencies. Skip if already installed.

In [1]:
import sys

!{sys.executable} -m pip install \
    numpy \
    matplotlib \
    seaborn \
    scikit-learn \
    sentence-transformers \
    bertopic \
    umap-learn \
    hdbscan \
    umap-learn

## 2. Imports

In [2]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
from umap import UMAP

from google.colab import drive

warnings.filterwarnings("ignore", category=DeprecationWarning)

drive.mount('/content/drive')


print("Imports OK.")

## 3. Configuration
Set all options here before running the rest of the notebook.

In [3]:
# ── Model Selection ────────────────────────────────────────────────────────────
MODEL_TYPE = 'lda'          # 'lda' | 'nmf' | 'bertopic'

# ── Data Source ────────────────────────────────────────────────────────────────
# Set INPUT_FILE to a path string to load from JSON, or None to load from DB.
INPUT_FILE = '/content/drive/MyDrive/ColabData/math_cs_mixed_s1.json'           # e.g. '/content/drive/MyDrive/dataset.json'

# ── Topic Count ────────────────────────────────────────────────────────────────
# For LDA/NMF: set an integer, or None to auto-detect from label count.
# For BERTopic: set an integer to fix K, or None for HDBSCAN auto-detection.
K_TOPICS = 15

# ── Concept Level ──────────────────────────────────────────────────────────────
TARGET_LEVEL = 0            # 0, 1, or 2

# ── Export ─────────────────────────────────────────────────────────────────────
EXPORT_HEATMAP = 'threshold_heatmap.png'

# ── BERTopic-only Options ──────────────────────────────────────────────────────
USE_APPROX_DIST      = True   # True  → c-TF-IDF approximate distribution
USE_LEMMATIZED_INPUT = False   # True  → lemmatize text before embedding

print(f"Config loaded: model={MODEL_TYPE}, K={K_TOPICS}, level={TARGET_LEVEL}")

## 4. Service Stubs
These lightweight wrappers call your existing `LDAService`, `NMFService`, and `BERTopicService`.
Replace the bodies with your actual service imports when running inside the Django project.

In [4]:
# ── If running INSIDE the Django project, replace the stubs below with: ────────
#
#   from api.services.lda_service     import LDAService
#   from api.services.nmf_service     import NMFService
#   from api.services.bertopic_service import BERTopicService
#
# ── Standalone stubs (sklearn-backed) ─────────────────────────────────────────

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation, NMF

academic_stopwords = [
    'finding', 'findings', 'illustrate', 'significant', 'provide', 'provides', 'potential', 'associated', 'effective', 'aspect', 'aspects', 'challenge', 'challenges',
    'paper', 'study', 'research', 'result', 'results', 'method', 'methodology',
    'proposed', 'propose', 'approach', 'based', 'using', 'used', 'use', 'to', 'we', 'source',
    'analysis', 'model', 'system', 'data', 'application', 'new', 'development',
    'performance', 'conclusion', 'abstract', 'introduction', 'work', 'time',
    'significant', 'shown', 'show', 'demonstrate', 'experiment', 'experimental',
    'university', 'department', 'author', 'et', 'al', 'figure', 'table',
    'high', 'low', 'large', 'small', 'different', 'various', 'property', 'properties', 'increase', 'effect', 'activity',
    'structure', 'compound', 'condition', 'quality', 'entry', 'contain', 'parameter', 'observe', 'report', 'present', 'evaluate'
        ]

full_stopwords = list(set(list(ENGLISH_STOP_WORDS) + academic_stopwords))

class LDAService:
    def __init__(self, n_topics=15):
        self.n_topics = n_topics
        self.vectorizer = CountVectorizer(stop_words=full_stopwords)
        self.model = LatentDirichletAllocation(n_components=n_topics, random_state=42)

    def fit_transform(self, documents):
        X = self.vectorizer.fit_transform(documents)
        return self.model.fit_transform(X)   # shape: (n_docs, n_topics)


class NMFService:
    def __init__(self, n_topics=15):
        self.n_topics = n_topics
        self.vectorizer = TfidfVectorizer(stop_words=full_stopwords)
        self.model = NMF(n_components=n_topics, random_state=42)

    def fit_transform(self, documents):
        X = self.vectorizer.fit_transform(documents)
        return self.model.fit_transform(X)   # shape: (n_docs, n_topics)


class BERTopicService:
    """
    Stub that wraps the real BERTopic library.
    Replace with your api.services.bertopic_service.BERTopicService if preferred.
    """
    def __init__(self, n_topics=None, use_approx_dist=True, use_lemmatized_input=False):
        self.n_topics = n_topics
        self.use_approx_dist = use_approx_dist
        self.use_lemmatized_input = use_lemmatized_input
    
    umap_model = UMAP(
            n_neighbors=15,
            n_components=5,
            min_dist=0.0, metric='cosine',
            random_state=42, low_memory=True
        )

    def fit_transform(self, documents):
        from bertopic import BERTopic
        nr_topics = self.n_topics if self.n_topics else None
        model = BERTopic(
            umap_model=self.umap_model,
            embedding_model="allenai/specter2_base",
            vectorizer_model=CountVectorizer(stop_words=full_stopwords),
            calculate_probabilities=not self.use_approx_dist,
            nr_topics=self.n_topics if self.n_topics else None,
            verbose=True,
            top_n_words=15)
        topics, probs = model.fit_transform(documents)
        return topics, probs


print("Service stubs ready.")

## 5. Load Data

In [5]:
target_key_hard  = f'true_label_l{TARGET_LEVEL}'
target_key_multi = f'multi_labels_l{TARGET_LEVEL}'

documents       = []
papers_data     = []
y_true_dominant = []

if INPUT_FILE:
    print(f"Loading data from: {INPUT_FILE}")
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        data = json.load(f)

    for item in data:
        text = item.get('text', '')
        if not text:
            text = f"{item.get('title', '')} {item.get('abstract', '')}"

        true_labels_set = set()
        top_label = item.get(target_key_hard)

        if target_key_multi in item and isinstance(item[target_key_multi], list):
            true_labels_set = set(item[target_key_multi])
        elif 'openalex_concepts' in item:
            valid = [c for c in item['openalex_concepts'] if c.get('level') == TARGET_LEVEL]
            true_labels_set.update([c['name'] for c in valid])
            if valid and not top_label:
                valid.sort(key=lambda x: x.get('score', 0), reverse=True)
                top_label = valid[0]['name']
        else:
            if top_label:
                true_labels_set.add(top_label)

        if true_labels_set and top_label and text.strip():
            documents.append(text)
            y_true_dominant.append(top_label)
            papers_data.append({'id': str(item.get('id', 'N/A')), 'true_labels': list(true_labels_set)})

    n_topics = K_TOPICS if K_TOPICS else len(set(y_true_dominant))

else:
    print("No INPUT_FILE set — loading from Django ORM (Paper model).")
    # ── Django ORM path ────────────────────────────────────────────────────────
    # Uncomment and adjust when running inside the Django project:
    #
    # import django, os
    # os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'your_project.settings')
    # django.setup()
    # from api.models import Paper
    #
    # papers = Paper.objects.exclude(abstract__isnull=True).exclude(abstract__exact='')
    # for paper in papers:
    #     text = f"{paper.title} {paper.abstract}"
    #     concepts  = paper.openalex_concepts
    #     valid = [c for c in concepts if c.get('level') == TARGET_LEVEL]
    #     true_labels_set = set([c['name'] for c in valid])
    #     if true_labels_set and text.strip():
    #         valid.sort(key=lambda x: x.get('score', 0), reverse=True)
    #         top_label = valid[0]['name']
    #         documents.append(text)
    #         y_true_dominant.append(top_label)
    #         papers_data.append({'id': str(paper.id), 'true_labels': list(true_labels_set)})
    # n_topics = K_TOPICS if K_TOPICS else len(set(y_true_dominant))
    raise RuntimeError("Set INPUT_FILE or uncomment the Django ORM block above.")

if not documents:
    raise ValueError("No documents found. Check your data source and label keys.")

print(f"Loaded {len(documents):,} documents | {len(set(y_true_dominant))} unique labels | n_topics={n_topics}")

## 6. Train Model & Extract Topic–Probability Matrix

In [6]:
y_pred_hard_ids  = []
doc_topic_matrix = None


if MODEL_TYPE == 'bertopic':
    actual_k = K_TOPICS  # None → HDBSCAN auto
    msg_k = actual_k if actual_k else None
    print(f"Training BERTopic with K={msg_k} topics...")

    service = BERTopicService(
        n_topics=actual_k,
        use_approx_dist=USE_APPROX_DIST,
        use_lemmatized_input=USE_LEMMATIZED_INPUT,
    )
    topics_hard, probs = service.fit_transform(documents)

    if probs is None:
        raise RuntimeError("BERTopic failed to generate probabilities.")
    
    probs = np.array(probs)

    if probs.ndim == 1:
        probs = probs.reshape(-1, 1)
        
    doc_topic_matrix = probs
    y_pred_hard_ids  = topics_hard
    n_topics = probs.shape[1]

elif MODEL_TYPE == 'lda':
    actual_k = K_TOPICS if K_TOPICS else len(set(y_true_dominant))
    print(f"Training LDA with K={actual_k} topics...")
    n_topics = actual_k

    service = LDAService(n_topics=actual_k)
    doc_topic_matrix = service.fit_transform(documents)
    y_pred_hard_ids  = np.argmax(doc_topic_matrix, axis=1)

elif MODEL_TYPE == 'nmf':
    actual_k = K_TOPICS if K_TOPICS else len(set(y_true_dominant))
    print(f"Training NMF with K={actual_k} topics...")
    n_topics = actual_k

    service = NMFService(n_topics=actual_k)
    matrix   = service.fit_transform(documents)
    row_sums = matrix.sum(axis=1)
    row_sums[row_sums == 0] = 1
    doc_topic_matrix = matrix / row_sums[:, np.newaxis]   # normalise → probabilities
    y_pred_hard_ids  = np.argmax(doc_topic_matrix, axis=1)

else:
    raise ValueError(f"Unknown MODEL_TYPE: '{MODEL_TYPE}'")

print(f"doc_topic_matrix shape: {np.array(doc_topic_matrix).shape}")

## 7. Build Cluster → Label Mapping

In [7]:
cluster_to_label_map = {}
unique_clusters = set(y_pred_hard_ids)

for cid in unique_clusters:
    indices = [i for i, x in enumerate(y_pred_hard_ids) if x == cid]
    if indices:
        labels_in_cluster = [y_true_dominant[i] for i in indices]
        cluster_to_label_map[cid] = Counter(labels_in_cluster).most_common(1)[0][0]
    else:
        cluster_to_label_map[cid] = 'Unknown'

print(f"Mapped {len(cluster_to_label_map)} clusters to labels.")
# Preview first 10 mappings
for k, v in list(cluster_to_label_map.items())[:10]:
    print(f"  Cluster {k:>4} → {v}")

## 8. Grid Search — Absolute × Relative Threshold

In [8]:
abs_thresholds = np.arange(0.05, 0.26, 0.05)   # [0.05, 0.10, 0.15, 0.20, 0.25]
rel_thresholds = np.arange(0.1,  0.6,  0.1)    # [0.1, 0.2, 0.3, 0.4, 0.5]

results_matrix = np.zeros((len(abs_thresholds), len(rel_thresholds)))

best_f1     = 0.0
best_params = (0.0, 0.0)

print("Running Grid Search...")

for i, abs_t in enumerate(abs_thresholds):
    for j, rel_t in enumerate(rel_thresholds):
        f1_list = []

        for doc_idx, paper_item in enumerate(papers_data):
            true_labels_set = set(paper_item['true_labels'])
            pred_labels     = set()
            probs           = doc_topic_matrix[doc_idx]

            max_prob = max(probs) if len(probs) > 0 else 0

            for t_id, prob in enumerate(probs):
                if prob > abs_t and prob >= (max_prob * rel_t):
                    mapped_label = cluster_to_label_map.get(t_id, 'Unknown')
                    if mapped_label != 'Unknown':
                        pred_labels.add(mapped_label)

            # Fallback: if no topic survived the filter, use the hard-assigned cluster
            hard_cluster_id = int(y_pred_hard_ids[doc_idx])
            if not pred_labels:
                pred_labels.add(cluster_to_label_map.get(hard_cluster_id, 'Unknown'))

            intersection = len(true_labels_set & pred_labels)
            p  = intersection / len(pred_labels)     if pred_labels     else 0
            r  = intersection / len(true_labels_set) if true_labels_set else 0
            f1 = 2 * p * r / (p + r)                if (p + r) > 0     else 0
            f1_list.append(f1)

        avg_f1 = np.mean(f1_list)
        results_matrix[i, j] = avg_f1

        if avg_f1 > best_f1:
            best_f1     = avg_f1
            best_params = (abs_t, rel_t)

print("Grid Search complete.")

## 9. Results Summary

In [9]:
mode_str = ''
if MODEL_TYPE == 'bertopic':
    mode_str = ' [APPROX (c-TF-IDF)]' if USE_APPROX_DIST else ' [HDBSCAN]'

title_str = f"{MODEL_TYPE.upper()}{mode_str}"

print('=' * 60)
print(f'GRID SEARCH RESULTS FOR {title_str}')
print('=' * 60)
print(f'Best F1-Score                                 : {best_f1:.4f}')
print(f'Optimal Absolute Threshold  (prob > X)        : {best_params[0]:.2f}')
print(f'Optimal Relative Threshold  (prob >= max * Y) : {best_params[1]:.2f}')
print('=' * 60)

## 10. Heatmap Visualisation

In [10]:
cmap = 'Purples' if MODEL_TYPE == 'bertopic' else 'YlGnBu'

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    results_matrix,
    annot=True,
    fmt='.4f',
    cmap=cmap,
    xticklabels=np.round(rel_thresholds, 2),
    yticklabels=np.round(abs_thresholds, 2),
    ax=ax,
)

ax.set_xlabel('Relative Threshold (Y%)', fontsize=12, fontweight='bold')
ax.set_ylabel('Absolute Threshold (X)',  fontsize=12, fontweight='bold')
ax.set_title(f'{title_str} Multi-label F1-Score Grid Search', fontsize=14, fontweight='bold')

# Highlight the best cell with a red border
best_i = np.where(np.isclose(abs_thresholds, best_params[0]))[0][0]
best_j = np.where(np.isclose(rel_thresholds, best_params[1]))[0][0]
ax.add_patch(plt.Rectangle((best_j, best_i), 1, 1, fill=False, edgecolor='red', lw=3))

plt.tight_layout()
plt.savefig(EXPORT_HEATMAP, dpi=300)
plt.show()
print(f"Heatmap saved to: {EXPORT_HEATMAP}")